# AC Campaign: Modern Queue 3: v3_full with tu-tf_efficientnetv2_s (lr=2e-4, Seed 42)
Kernel slug: `ac-tf-efficientnetv2-s-s42`
Part of the AC 5-Fold Cross-Validation Campaign on Kaggle T4 GPU.

In [ ]:
# First-Cell Pre-flight Sanity Check
import os
import sys
import subprocess
import torch

print("=== Pre-flight Sanity Check ===")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("PRE-FLIGHT CHECK FAILED: CUDA GPU is not available! Please enable GPU accelerator in Kaggle settings.")
print("GPU device:", torch.cuda.get_device_name(0))

# Ensure required libraries are installed
try:
    import segmentation_models_pytorch as smp
    import timm
except ImportError:
    print("Installing segmentation-models-pytorch and timm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "segmentation-models-pytorch", "timm"])
    import segmentation_models_pytorch as smp
    import timm

print(f"SMP version: {smp.__version__} | timm version: {timm.__version__}")

# Verify encoder initialization and pretrained weights loading
encoder_name = "tu-tf_efficientnetv2_s"
_weights = True if encoder_name.startswith("tu-") else "imagenet"
try:
    print(f"Testing initialization of smp.Unet with encoder '{encoder_name}' (encoder_weights={_weights})...")
    _m = smp.Unet(encoder_name=encoder_name, encoder_weights=_weights, in_channels=3, classes=4)
    _dummy_x = torch.randn(1, 3, 384, 384)
    _dummy_out = _m(_dummy_x)
    assert _dummy_out.shape == (1, 4, 384, 384), f"Unexpected output shape: {_dummy_out.shape}"
    del _m, _dummy_x, _dummy_out
    print(f"PRE-FLIGHT CHECK PASSED: '{encoder_name}' initialized and verified successfully.")
except Exception as exc:
    raise RuntimeError(f"PRE-FLIGHT CHECK FAILED: Failed to initialize encoder '{encoder_name}': {exc}")

In [ ]:
# Setup code from fetal-code-snapshot
import os
import shutil
import sys
from pathlib import Path

code_input = None
candidates = [
    Path("/kaggle/input/fetal-code-snapshot"),
    Path("/kaggle/input/datasets/prantakumerpandit390/fetal-code-snapshot"),
]
for c in candidates:
    if c.exists() and (c / "COMMIT.txt").exists():
        code_input = c
        break

if code_input is None and Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").rglob("COMMIT.txt"):
        code_input = p.parent
        break

if code_input is None:
    raise FileNotFoundError("Could not find fetal-code-snapshot in /kaggle/input")

print("Found code snapshot at:", code_input)
work_dir = Path("/kaggle/working/code")
if not work_dir.exists():
    shutil.copytree(code_input, work_dir)

# Unzip nested directories if necessary
for z in work_dir.glob("*.zip"):
    dir_name = z.stem
    if not (work_dir / dir_name).is_dir():
        print(f"Extracting {z.name} -> {dir_name}/")
        shutil.unpack_archive(str(z), str(work_dir / dir_name))

os.chdir(work_dir)
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

print("Working directory set to:", os.getcwd())
commit_p = work_dir / "COMMIT.txt"
if commit_p.exists():
    print("Code Commit:", commit_p.read_text().strip())

In [ ]:
# Install required dependencies
!pip install --quiet segmentation-models-pytorch timm

In [ ]:
# Setup AC Dataset
import shutil
from pathlib import Path

data_candidate = None
for c in [
    Path("/kaggle/input/ac-fetal-v1"),
    Path("/kaggle/input/datasets/prantakumerpandit390/ac-fetal-v1"),
]:
    if c.exists():
        data_candidate = c
        break

if data_candidate is None and Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").rglob("folds_v1.csv"):
        data_candidate = p.parent
        break

if data_candidate is None:
    raise FileNotFoundError("Could not find ac-fetal-v1 dataset in /kaggle/input")

print("Found AC dataset at:", data_candidate)

# Setup target data directory
data_dir = Path("/kaggle/working/data")
data_dir.mkdir(parents=True, exist_ok=True)
os.environ["FETAL_DATA_DIR"] = str(data_dir)

# Unpack FP and UCL if zipped, or symlink/copy if directory
for src_name in ["FP", "UCL"]:
    zip_p = data_candidate / f"{src_name}.zip"
    folder_p = data_candidate / src_name
    dest_p = data_dir / src_name

    if not dest_p.exists():
        if zip_p.exists():
            print(f"Unpacking {zip_p} -> {dest_p}...")
            shutil.unpack_archive(str(zip_p), str(dest_p))
        elif folder_p.is_dir():
            print(f"Copying {folder_p} -> {dest_p}...")
            shutil.copytree(str(folder_p), str(dest_p))

# Copy folds_v1.csv
folds_src = data_candidate / "folds_v1.csv"
if not (data_dir / "folds_v1.csv").exists() and folds_src.exists():
    shutil.copy2(str(folds_src), str(data_dir / "folds_v1.csv"))

print("Data directory contents:", list(data_dir.glob("*")))
print("FP images count:", len(list((data_dir / "FP" / "images" / "Abdomen").glob("*.png"))))
print("UCL images count:", len(list((data_dir / "UCL" / "images" / "Abdomen").glob("*.*"))))
print("folds_v1.csv exists:", (data_dir / "folds_v1.csv").exists())

In [ ]:
# Run 5-fold AC Cross-Validation
import subprocess

cmd = """python scripts/04_train_ac.py --fold all --data-dir /kaggle/working/data --folds-csv /kaggle/working/data/folds_v1.csv --out-dir /kaggle/working/ac-tf-efficientnetv2-s-s42 --seed 42 --recipe v3_full --encoder tu-tf_efficientnetv2_s --lr 2e-4"""
print("Running command:\n", cmd)
ret = subprocess.run(cmd, shell=True)
if ret.returncode != 0:
    raise RuntimeError(f"Training failed with return code {ret.returncode}")

In [ ]:
# Display Final OOF Summary
import json
from pathlib import Path
import pandas as pd

summary_p = Path("/kaggle/working/ac-tf-efficientnetv2-s-s42/oof_summary.json")
if summary_p.exists():
    with open(summary_p, "r") as f:
        summary = json.load(f)
    print("=== Pooled 5-Fold OOF Summary ===")
    print(json.dumps(summary, indent=2))

oof_csv = Path("/kaggle/working/ac-tf-efficientnetv2-s-s42/oof_predictions_all.csv")
if oof_csv.exists():
    df_oof = pd.read_csv(oof_csv)
    print(f"\nOOF Predictions CSV: {oof_csv} (rows={len(df_oof)})")
    print(df_oof.head())